# FoundationStereo: E1, E2 and LoRA fine-tuning (Protocol A)

Follow-up to the E0 run of `foundationstereo_kitti_finetuning.ipynb` (v1). Same setup, same downloads, same Protocol A
split, same loss / optimizer / LR scheduler / early stopping. Four new runs:

| Run | What is trained | BatchNorm statistics | LR | Question |
|---|---|---|---|---|
| E0 (done, reference) | everything except DepthAnything | updating (drift) | 1e-5 | baseline |
| **E1_bn_locked** | everything except DepthAnything | **locked** | 1e-5 | How much of the Middlebury drop is BatchNorm drift? |
| **E2_gru_only** | GRU update block + upsampler | locked | 1e-5 | Can refinement alone adapt to KITTI? |
| **E3a_lora_all** | LoRA r=8 on every conv/linear outside DepthAnything | locked | 1e-4 | Low-rank version of E1 |
| **E3b_lora_gru** | LoRA r=8 on GRU update block + upsampler | locked | 1e-4 | Low-rank version of E2 |

**Protocol A**: train all 194 KITTI 2012 pairs, validate on 40 KITTI 2015 pairs, test on the other 160 KITTI 2015 pairs.
Every run is also evaluated on Middlebury trainingH (half resolution, non-occluded, BP-2).
Zero-shot and E0 numbers come from the earlier runs (same split, same code) and are printed next to every new result.

### How to run
1. Upload **this notebook** and **`foundationstereo_source_bundle.zip`** to Colab
   (or upload `foundationstereo_e1_e2_lora_bundle.zip`; the notebook finds the source zip inside it).
2. Runtime → Change runtime type → **GPU** (A100 or L4 recommended).
3. Runtime → **Run all**. Allow Google Drive access when asked: results and checkpoints are saved there after every run,
   so if the session disconnects you can **Run all** again and finished runs are skipped.

In [ ]:
# ===== 1. Configuration =====
from pathlib import Path

EXPERIMENTS_TO_RUN = ["E1_bn_locked", "E2_gru_only", "E3a_lora_all", "E3b_lora_gru"]

# Training recipe: identical to E0 (Protocol A)
SEED = 2025
MAX_STEPS = 5000
EVAL_INTERVAL = 50              # validate every 50 steps
LOG_INTERVAL = 50               # print training loss every 50 steps
SCHEDULER_PATIENCE = 2          # ReduceLROnPlateau: halve LR after 2 validations without improvement
EARLY_STOPPING_PATIENCE = 3     # stop after 3 validations without improvement
FULL_FT_LR = 1e-5

# LoRA
LORA_RANK = 8
LORA_ALPHA = 16
LORA_LR = 1e-4                  # LoRA B matrices start at zero; 1e-5 would barely move them

# Saving
USE_DRIVE = True                # save results + checkpoints to Google Drive (survives disconnects)
DRIVE_DIR = Path("/content/drive/MyDrive/foundationstereo_e1_e4_lora")
SAVE_CHECKPOINTS = True         # full model per run (~1.5 GB, LoRA merged in) + small LoRA adapter files
DOWNLOAD_FULL_CHECKPOINTS = False   # also push each 1.5 GB checkpoint through the browser at the end

print("Experiments:", EXPERIMENTS_TO_RUN)
print(f"Recipe: max {MAX_STEPS} steps, validate every {EVAL_INTERVAL}, LR scheduler patience {SCHEDULER_PATIENCE}, "
      f"early stopping patience {EARLY_STOPPING_PATIENCE}, seed {SEED}")
print(f"Full fine-tuning LR {FULL_FT_LR:g} | LoRA rank {LORA_RANK}, alpha {LORA_ALPHA}, LR {LORA_LR:g}")

In [ ]:
# ===== 2. Colab setup (same as v1) =====
!pip -q install omegaconf open3d imageio opencv-python einops timm scipy scikit-image gdown ruamel.yaml transformations trimesh joblib pandas

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc
import json
import math
import shutil
import sys
import time
import zipfile

import numpy as np
import torch

# Source code: foundationstereo_source_bundle.zip, either uploaded directly or inside the bundle zip
SOURCE_ZIP = Path("/content/foundationstereo_source_bundle.zip")
BUNDLE_UPLOAD = Path("/content/foundationstereo_e1_e2_lora_bundle.zip")
if not SOURCE_ZIP.exists() and BUNDLE_UPLOAD.exists():
    with zipfile.ZipFile(BUNDLE_UPLOAD) as archive:
        inner = [n for n in archive.namelist() if n.endswith("foundationstereo_source_bundle.zip")]
        if inner:
            SOURCE_ZIP.write_bytes(archive.read(inner[0]))
            print("Took foundationstereo_source_bundle.zip from", BUNDLE_UPLOAD.name)
if not SOURCE_ZIP.exists():
    raise FileNotFoundError("Upload foundationstereo_source_bundle.zip (or foundationstereo_e1_e2_lora_bundle.zip) to /content first.")

BUNDLE = Path("/content/foundationstereo_bundle")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    archive.extractall(BUNDLE)
SOURCE_ROOT = next((p.parent.parent for p in BUNDLE.rglob("core/foundation_stereo.py")), None)
if SOURCE_ROOT is None:
    raise FileNotFoundError("The source bundle does not contain core/foundation_stereo.py.")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

ROOT = Path("/content/foundationstereo_kitti")
DATA = ROOT / "data"
CHECKPOINT = ROOT / "checkpoints" / "23-51-11"
OUTPUT = ROOT / "outputs_e1_e2_lora"
for folder in [DATA, CHECKPOINT, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Google Drive for results + checkpoints
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        DRIVE_OK = True
    except Exception as error:
        print("Google Drive not available, saving locally only:", error)
RESULTS_DIR = DRIVE_DIR if DRIVE_OK else OUTPUT
CKPT_DIR = RESULTS_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("Source root:", SOURCE_ROOT)
print("Results folder:", RESULTS_DIR, "(Google Drive)" if DRIVE_OK else "(local: download before the session ends)")
print("Checkpoint folder:", CKPT_DIR)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before continuing.")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

In [ ]:
# ===== 3. Download and extract KITTI 2012/2015 training data =====
import requests

def zip_ok(path):
    # True if every entry of the archive can be read (catches truncated / corrupted downloads)
    try:
        with zipfile.ZipFile(path) as archive:
            return archive.testzip() is None
    except (zipfile.BadZipFile, OSError):
        return False

def download(url, path, attempts=3):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and zip_ok(path):
        print("Already downloaded:", path.name)
        return
    for attempt in range(1, attempts + 1):
        print(f"Downloading: {path.name} (attempt {attempt}/{attempts})", flush=True)
        partial = path.with_suffix(path.suffix + ".part")
        try:
            with requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True, timeout=120) as response:
                response.raise_for_status()
                with partial.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)
            partial.replace(path)
        except requests.RequestException as error:
            print("  download failed:", error)
            continue
        if zip_ok(path):
            print(f"  done: {path.stat().st_size / 1024**2:.0f} MiB", flush=True)
            return
        print("  archive is corrupted, downloading again")
        path.unlink()
    raise RuntimeError(f"Could not download a valid {path.name} from {url}")

def extract_once(archive_path, destination):
    destination = Path(destination)
    marker = destination / ".extracted"
    if marker.exists():
        print("Already extracted:", destination.name)
        return
    if destination.exists():
        shutil.rmtree(destination)   # remove a half-finished extraction
    destination.mkdir(parents=True)
    print("Extracting:", Path(archive_path).name, flush=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)
    marker.write_text("ok")

KITTI12 = DATA / "kitti2012"
KITTI15 = DATA / "kitti2015"
download("https://s3.eu-central-1.amazonaws.com/avg-kitti/data_stereo_flow.zip", DATA / "data_stereo_flow.zip")
download("https://s3.eu-central-1.amazonaws.com/avg-kitti/data_scene_flow.zip", DATA / "data_scene_flow.zip")
extract_once(DATA / "data_stereo_flow.zip", KITTI12)
extract_once(DATA / "data_scene_flow.zip", KITTI15)

def find_training_root(root, dataset):
    image_folder = "colored_0" if dataset == "kitti2012" else "image_2"
    candidates = [p.parent.parent for p in Path(root).rglob(image_folder) if p.is_dir() and p.parent.name == "training"]
    if not candidates:
        raise FileNotFoundError(f"Could not find training/{image_folder} under {root}")
    return candidates[0]

KITTI12_ROOT = find_training_root(KITTI12, "kitti2012")
KITTI15_ROOT = find_training_root(KITTI15, "kitti2015")
print("KITTI 2012:", KITTI12_ROOT)
print("KITTI 2015:", KITTI15_ROOT)

In [ ]:
# ===== 4. Download the official FoundationStereo checkpoint (same as v1) =====
from huggingface_hub import hf_hub_download

MODEL_FILE = CHECKPOINT / "model_best_bp2.pth"
CFG_FILE = CHECKPOINT / "cfg.yaml"
HF_REPO = "steve-redefine/FoundationStereoWeights"

if not CFG_FILE.exists() or CFG_FILE.stat().st_size < 100:
    print("Downloading cfg.yaml from Hugging Face", flush=True)
    hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename="23-51-11/cfg.yaml", local_dir=CHECKPOINT.parent)
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    print("Downloading model_best_bp2.pth from Hugging Face (about 3.3 GiB)", flush=True)
    hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename="23-51-11/model_best_bp2.pth", local_dir=CHECKPOINT.parent)
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 1_000_000_000:
    raise FileNotFoundError("The FoundationStereo checkpoint was not downloaded completely. Delete the file and rerun this cell.")
print("Checkpoint:", MODEL_FILE)
print("Size GiB:", round(MODEL_FILE.stat().st_size / 1024**3, 3))

In [ ]:
# ===== 5. Dataset indexing and the Protocol A split (identical to v1) =====
from dataclasses import dataclass
import imageio.v2 as imageio

@dataclass(frozen=True)
class KITTISample:
    dataset: str
    index: str
    root: Path

    @property
    def image_folder(self):
        return "colored_0" if self.dataset == "kitti2012" else "image_2"

    @property
    def right_folder(self):
        return "colored_1" if self.dataset == "kitti2012" else "image_3"

    def frame_name(self, folder):
        directory = self.root / "training" / folder
        candidates = [directory / f"{self.index}{suffix}.png" for suffix in ["_10", ""]]
        if self.dataset == "kitti2015":
            candidates.extend(sorted(directory.glob(f"{self.index}_*.png")))
        for candidate in candidates:
            if candidate.exists():
                return candidate.name
        raise FileNotFoundError(f"No frame found for {self.dataset}/{self.index} in {directory}")

    @property
    def left(self):
        return self.root / "training" / self.image_folder / self.frame_name(self.image_folder)

    @property
    def right(self):
        return self.root / "training" / self.right_folder / self.frame_name(self.right_folder)

    @property
    def disp_occ(self):
        folder = "disp_occ_0" if self.dataset == "kitti2015" else "disp_occ"
        return self.root / "training" / folder / self.frame_name(folder)

    @property
    def disp_noc(self):
        folder = "disp_noc_0" if self.dataset == "kitti2015" else "disp_noc"
        return self.root / "training" / folder / self.frame_name(folder)

def index_kitti(dataset, root):
    image_folder = "colored_0" if dataset == "kitti2012" else "image_2"
    indices = sorted({p.stem.split("_", 1)[0] for p in (Path(root) / "training" / image_folder).glob("*.png")})
    samples = [KITTISample(dataset, index, Path(root)) for index in indices]
    for sample in samples:
        for path in [sample.left, sample.right, sample.disp_occ, sample.disp_noc]:
            if not path.exists():
                raise FileNotFoundError(path)
    return samples

kitti12_samples = index_kitti("kitti2012", KITTI12_ROOT)
kitti15_samples = index_kitti("kitti2015", KITTI15_ROOT)
print("KITTI 2012 pairs:", len(kitti12_samples))
print("KITTI 2015 pairs:", len(kitti15_samples))
if len(kitti12_samples) != 194 or len(kitti15_samples) != 200:
    raise RuntimeError("Unexpected KITTI training-pair count.")

# Same generator and the same two draws as v1, so Protocol A is exactly the E0 split
rng = np.random.default_rng(2025)
k15_order = rng.permutation(len(kitti15_samples))
k12_order = rng.permutation(len(kitti12_samples))
protocol_a = {
    "train": kitti12_samples,
    "val": [kitti15_samples[i] for i in k15_order[:40]],
    "test": [kitti15_samples[i] for i in k15_order[40:]],
}
print(f"Protocol A: train={len(protocol_a['train'])}, val={len(protocol_a['val'])}, test={len(protocol_a['test'])}")
# Check against the E0 log ("Inference 1/40 (kitti2015/000139)", "Inference 10/40 (kitti2015/000044)", ...)
split_check = {"val #1": (protocol_a["val"][0].index, "000139"), "val #10": (protocol_a["val"][9].index, "000044"),
               "test #1": (protocol_a["test"][0].index, "000129"), "test #10": (protocol_a["test"][9].index, "000008")}
for label, (got, expected) in split_check.items():
    print(f"  {label}: {got} (E0: {expected}) {'OK' if got == expected else 'MISMATCH'}")
if any(got != expected for got, expected in split_check.values()):
    print("WARNING: split differs from E0, results will not be directly comparable")
else:
    print("Protocol A split identical to E0")

def load_kitti(sample):
    left = imageio.imread(sample.left)
    right = imageio.imread(sample.right)
    occ = imageio.imread(sample.disp_occ).astype(np.float32) / 256.0
    noc = imageio.imread(sample.disp_noc).astype(np.float32) / 256.0
    return left, right, occ, noc

def random_crop(left, right, disp, height=256, width=512, rng=None):
    rng = rng or np.random.default_rng()
    h, w = left.shape[:2]
    if h < height or w < width:
        raise ValueError(f"Image {left.shape[:2]} is smaller than crop {(height, width)}")
    y = int(rng.integers(0, h - height + 1))
    x = int(rng.integers(0, w - width + 1))
    return left[y:y+height, x:x+width], right[y:y+height, x:x+width], disp[y:y+height, x:x+width]

In [ ]:
# ===== 6. Download Middlebury half-resolution data (same as v1) =====
MIDDLEBURY = DATA / "middlebury"
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-data-H.zip", MIDDLEBURY / "data-H.zip")
download("https://vision.middlebury.edu/stereo/submit3/zip/MiddEval3-GT0-H.zip", MIDDLEBURY / "gt0-H.zip")
extract_once(MIDDLEBURY / "data-H.zip", MIDDLEBURY / "data-H")
extract_once(MIDDLEBURY / "gt0-H.zip", MIDDLEBURY / "gt0-H")
MIDDLEBURY_DATA = next(MIDDLEBURY.rglob("data-H/MiddEval3"))
MIDDLEBURY_GT = next(MIDDLEBURY.rglob("gt0-H/MiddEval3"))
middlebury_scenes = sorted(p.name for p in (MIDDLEBURY_DATA / "trainingH").iterdir() if p.is_dir())
print(f"Middlebury trainingH scenes ({len(middlebury_scenes)}):", ", ".join(middlebury_scenes))
if len(middlebury_scenes) != 15:
    raise RuntimeError("Expected 15 Middlebury trainingH scenes.")

def read_pfm(path):
    with open(path, "rb") as file:
        if file.readline().strip() != b"Pf":
            raise ValueError(path)
        width, height = map(int, file.readline().split())
        scale = float(file.readline())
        endian = "<" if scale < 0 else ">"
        values = np.fromfile(file, endian + "f4")
    return np.flipud(values.reshape(height, width)).astype(np.float32)

In [ ]:
# ===== 7. Freezing, BatchNorm locking and LoRA =====
import torch.nn as nn

# Trained in E2 / wrapped with LoRA in E3b: ConvGRU refinement + learned upsampler
GRU_MODULES = ["update_block", "spx_2_gru", "spx_gru", "stem_2"]

def set_trainable(model, trainable_children=None):
    # None -> every top-level module (as in E0/E1); a list -> only those; [] -> nothing.
    # DepthAnything (feature.dino) is always frozen.
    for name, child in model.named_children():
        flag = trainable_children is None or name in trainable_children
        for parameter in child.parameters():
            parameter.requires_grad = flag
    for parameter in model.feature.dino.parameters():
        parameter.requires_grad = False

def lock_batchnorm(model):
    # BatchNorm uses its stored running statistics and stops updating them (no drift toward KITTI)
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()

def set_train_mode(model):
    model.train()
    model.feature.dino.eval()
    lock_batchnorm(model)

class LoRAConv(nn.Module):
    # y = W x + (alpha / r) * B(A(x)).  A copies the conv's kernel/stride/padding, B is 1x1 and starts at zero,
    # so training starts exactly from the pretrained model.
    def __init__(self, base, rank, alpha):
        super().__init__()
        conv = type(base)
        self.base = base
        self.lora_A = conv(base.in_channels, rank, base.kernel_size, stride=base.stride, padding=base.padding,
                           dilation=base.dilation, bias=False, padding_mode=base.padding_mode)
        self.lora_B = conv(rank, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.scale = alpha / rank
        self.to(base.weight.device)

    @property
    def weight(self):   # some timm layers read .weight / .bias of their convs directly
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(x))

    @torch.no_grad()
    def merged(self):
        delta = self.lora_B.weight.flatten(1) @ self.lora_A.weight.flatten(1)
        self.base.weight += self.scale * delta.view_as(self.base.weight).to(self.base.weight.dtype)
        return self.base

class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.scale = alpha / rank
        self.to(base.weight.device)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(x))

    @torch.no_grad()
    def merged(self):
        self.base.weight += self.scale * (self.lora_B.weight @ self.lora_A.weight).to(self.base.weight.dtype)
        return self.base

def inject_lora(model, scope=None, rank=8, alpha=16):
    # Wrap every plain Conv2d/Conv3d (groups == 1) and Linear outside DepthAnything.
    # scope=None -> whole model; otherwise only inside these top-level modules.
    # Transposed and depthwise convs stay frozen.
    targets = []
    for parent_name, parent in model.named_modules():
        for child_name, child in parent.named_children():
            full = f"{parent_name}.{child_name}" if parent_name else child_name
            if full.startswith("feature.dino"):
                continue
            if scope is not None and full.split(".")[0] not in scope:
                continue
            if type(child) in (nn.Conv2d, nn.Conv3d) and child.groups == 1:
                targets.append((parent, child_name, full, LoRAConv))
            elif type(child) is nn.Linear:
                targets.append((parent, child_name, full, LoRALinear))
    for parent, child_name, _, wrapper in targets:
        setattr(parent, child_name, wrapper(getattr(parent, child_name), rank, alpha))
    return [full for _, _, full, _ in targets]

def merge_lora(model):
    # Fold LoRA into the base weights -> a plain FoundationStereo state_dict (loads like any checkpoint)
    replacements = [(parent, name, child)
                    for _, parent in model.named_modules()
                    for name, child in parent.named_children()
                    if isinstance(child, (LoRAConv, LoRALinear))]
    for parent, name, child in replacements:
        setattr(parent, name, child.merged())
    return len(replacements)

def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

# Self-test (CPU, tiny layers): a merged LoRA layer must reproduce the wrapped layer exactly
torch.manual_seed(0)
for base, x in [(nn.Conv2d(6, 10, 3, stride=2, padding=1), torch.randn(2, 6, 9, 11)),
                (nn.Conv3d(4, 5, 3, padding=1), torch.randn(1, 4, 5, 6, 7)),
                (nn.Linear(7, 9), torch.randn(3, 7))]:
    wrapped = (LoRALinear if isinstance(base, nn.Linear) else LoRAConv)(base, 4, 8)
    nn.init.normal_(wrapped.lora_B.weight)
    with torch.no_grad():
        expected = wrapped(x)
        actual = wrapped.merged()(x)
    assert torch.allclose(expected, actual, atol=1e-5), f"LoRA merge mismatch for {type(base).__name__}"
print("LoRA merge self-test passed (Conv2d, Conv3d, Linear)")

In [ ]:
# ===== 8. Model, fine-tuning loop (LR scheduler + early stopping) and evaluation =====
from omegaconf import OmegaConf
from core.foundation_stereo import FoundationStereo
from core.utils.utils import InputPadder

def make_base_model():
    cfg = OmegaConf.load(CHECKPOINT / "cfg.yaml")
    cfg["vit_size"] = "vitl"
    cfg["mixed_precision"] = True
    cfg["valid_iters"] = 32
    model = FoundationStereo(cfg)
    state = torch.load(MODEL_FILE, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"])
    return model

def build_experiment_model(spec):
    model = make_base_model()
    if spec["kind"] == "full":
        set_trainable(model, spec["trainable"])
        lora_layers = []
    else:
        set_trainable(model, [])
        lora_layers = inject_lora(model, spec["scope"], LORA_RANK, LORA_ALPHA)
    model.cuda()
    model.eval()
    return model, lora_layers

def disparity_loss(predictions, target, valid, gamma=0.9):
    # Same sequence loss as v1: smooth-L1 on every GRU iteration, later iterations weighted more
    weights = np.asarray([gamma ** (len(predictions) - i - 1) for i in range(len(predictions))], dtype=np.float32)
    weights = weights / weights.sum()
    loss = target.new_zeros(())
    for weight, prediction in zip(weights, predictions):
        error = torch.abs(prediction[:, 0] - target)
        smooth = torch.where(error < 1.0, 0.5 * error.square(), error - 0.5)
        loss = loss + float(weight) * smooth[valid].mean()
    return loss

def batch_from_sample(sample, crop_rng):
    left, right, occ, _ = load_kitti(sample)
    left, right, occ = random_crop(left, right, occ, rng=crop_rng)
    x0 = torch.from_numpy(np.ascontiguousarray(left)).cuda().float().permute(2, 0, 1)[None]
    x1 = torch.from_numpy(np.ascontiguousarray(right)).cuda().float().permute(2, 0, 1)[None]
    target = torch.from_numpy(np.ascontiguousarray(occ)).cuda().float()[None]
    valid = torch.isfinite(target) & (target > 0)
    return x0, x1, target, valid

def predict(model, left, right, iters=32):
    h, w = left.shape[:2]
    x0 = torch.from_numpy(left).cuda().float().permute(2, 0, 1)[None]
    x1 = torch.from_numpy(right).cuda().float().permute(2, 0, 1)[None]
    padder = InputPadder(x0.shape, divis_by=32, force_square=False)
    x0, x1 = padder.pad(x0, x1)
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=True):
        disp = model.forward(x0, x1, iters=iters, test_mode=True)
    return padder.unpad(disp.float()).detach().cpu().numpy().reshape(h, w)

def kitti_metrics(model, samples, label=None):
    # Same metrics as v1: EPE and D1 (error > 3 px and > 5 %), on disp_occ ("all") and disp_noc ("noc")
    model.eval()
    rows = []
    if label:
        print(f"Starting {label} inference on {len(samples)} samples...", flush=True)
    for number, sample in enumerate(samples, start=1):
        if label and (number == 1 or number % 10 == 0 or number == len(samples)):
            print(f"Inference {number}/{len(samples)} ({sample.dataset}/{sample.index})", flush=True)
        left, right, occ, noc = load_kitti(sample)
        prediction = predict(model, left, right)
        for mask, gt in [("all", occ), ("noc", noc)]:
            valid = np.isfinite(prediction) & np.isfinite(gt) & (gt > 0)
            error = np.abs(prediction[valid] - gt[valid])
            bad = (error > 3.0) & (error / np.maximum(gt[valid], 1e-6) > 0.05)
            rows.append({"dataset": sample.dataset, "index": sample.index, "mask": mask,
                         "valid_pixels": int(valid.sum()), "epe": float(error.mean()),
                         "bad_pixels": int(bad.sum()), "d1": float(bad.mean() * 100.0)})
    return rows

def aggregate_kitti(rows):
    out = {}
    for mask in ["all", "noc"]:
        subset = [r for r in rows if r["mask"] == mask]
        total_valid = sum(r["valid_pixels"] for r in subset)
        out[mask] = {
            "epe": round(sum(r["epe"] * r["valid_pixels"] for r in subset) / total_valid, 4),
            "d1": round(100.0 * sum(r["bad_pixels"] for r in subset) / total_valid, 4),
            "mean_image_d1": round(float(np.mean([r["d1"] for r in subset])), 4),
        }
    return out

def summarize(summary, label):
    # Same printout format as v1
    print(f"\n{label}")
    for mask in ["all", "noc"]:
        m = summary[mask]
        print(mask, "weighted EPE", f"{m['epe']:.4f}", "global D1", f"{m['d1']:.4f}%", "mean-image D1", f"{m['mean_image_d1']:.4f}%")

def middlebury_metrics(model, label):
    model.eval()
    rows = []
    print(f"Starting {label} Middlebury inference on {len(middlebury_scenes)} scenes...", flush=True)
    for number, scene in enumerate(middlebury_scenes, start=1):
        left = imageio.imread(MIDDLEBURY_DATA / "trainingH" / scene / "im0.png")
        right = imageio.imread(MIDDLEBURY_DATA / "trainingH" / scene / "im1.png")
        gt = read_pfm(MIDDLEBURY_GT / "trainingH" / scene / "disp0GT.pfm")
        mask = imageio.imread(MIDDLEBURY_GT / "trainingH" / scene / "mask0nocc.png") == 255
        prediction = predict(model, left, right)
        valid = np.isfinite(prediction) & np.isfinite(gt) & mask
        error = np.abs(prediction[valid] - gt[valid])
        bad = error > 2.0
        rows.append({"scene": scene, "valid_pixels": int(valid.sum()), "bad_pixels": int(bad.sum()),
                     "epe": float(error.mean()), "bp2": float(bad.mean() * 100.0)})
        print(f"Middlebury {number:2d}/{len(middlebury_scenes)} {scene:<12} EPE {rows[-1]['epe']:.4f}  BP-2 {rows[-1]['bp2']:.4f}%", flush=True)
    return rows

def aggregate_middlebury(rows):
    total_valid = sum(r["valid_pixels"] for r in rows)
    return {
        "epe": round(sum(r["epe"] * r["valid_pixels"] for r in rows) / total_valid, 4),
        "bp2": round(100.0 * sum(r["bad_pixels"] for r in rows) / total_valid, 4),
        "mean_scene_bp2": round(float(np.mean([r["bp2"] for r in rows])), 4),
    }

def validation_d1_noc(model, samples):
    noc = aggregate_kitti(kitti_metrics(model, samples))["noc"]
    return noc["d1"]

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def finetune(model, train_samples, val_samples, lr, seed):
    # Same recipe as v1: AdamW, gradient clipping 1.0, AMP, ReduceLROnPlateau on validation D1-noc,
    # early stopping, best validation weights restored at the end.
    torch.manual_seed(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE, threshold=1e-3, min_lr=1e-7)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    crop_rng = np.random.default_rng(seed)   # same crop sequence as E0

    print("Validation before training (step 0)...", flush=True)
    start_d1 = validation_d1_noc(model, val_samples)
    print(f"validation step 0: D1-noc {start_d1:.4f}% (starting point)", flush=True)
    best = {"d1": start_d1, "step": 0, "state": trainable_state(model)}
    curve = {"0": start_d1}
    loss_curve, losses, lr_changes = {}, [], []
    stale, stopped_at, skipped = 0, None, 0
    started = time.time()
    set_train_mode(model)

    for step in range(1, MAX_STEPS + 1):
        sample = train_samples[int(crop_rng.integers(0, len(train_samples)))]
        x0, x1, target, valid = batch_from_sample(sample, crop_rng)
        if valid.sum() == 0:
            skipped += 1
        else:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=True):
                _, predictions = model.forward(x0, x1, iters=12, test_mode=False)
                loss = disparity_loss(predictions, target, valid)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        if step % LOG_INTERVAL == 0:
            recent = float(np.mean(losses[-LOG_INTERVAL:])) if losses else float("nan")
            loss_curve[str(step)] = round(recent, 5)
            elapsed = time.time() - started
            print(f"step {step}/{MAX_STEPS}, loss {recent:.5f}, lr {optimizer.param_groups[0]['lr']:.2e}, "
                  f"{elapsed / step:.2f} s/step, elapsed {elapsed / 60:.1f} min", flush=True)

        if step % EVAL_INTERVAL == 0 or step == MAX_STEPS:
            val_d1 = validation_d1_noc(model, val_samples)
            curve[str(step)] = val_d1
            old_lr = optimizer.param_groups[0]["lr"]
            scheduler.step(val_d1)
            new_lr = optimizer.param_groups[0]["lr"]
            if val_d1 < best["d1"]:
                best = {"d1": val_d1, "step": step, "state": trainable_state(model)}
                stale = 0
                status = "new best, weights kept"
            else:
                stale += 1
                status = f"no improvement ({stale}/{EARLY_STOPPING_PATIENCE}), best {best['d1']:.4f}% at step {best['step']}"
            print(f"validation step {step}: D1-noc {val_d1:.4f}%, lr {new_lr:.2e} | {status}", flush=True)
            if new_lr < old_lr:
                lr_changes.append({"step": step, "from": old_lr, "to": new_lr})
                print(f"LR scheduler: learning rate reduced {old_lr:.2e} -> {new_lr:.2e}", flush=True)
            if stale >= EARLY_STOPPING_PATIENCE:
                stopped_at = step
                print(f"early stopping at step {step}; no validation improvement for {stale} evaluations", flush=True)
                break
            set_train_mode(model)

    if stopped_at is None:
        stopped_at = MAX_STEPS
        print(f"reached MAX_STEPS={MAX_STEPS} without early stopping", flush=True)
    model.load_state_dict(best["state"], strict=False)
    model.eval()
    minutes = (time.time() - started) / 60
    print(f"Restored best weights from step {best['step']} (validation D1-noc {best['d1']:.4f}%). Training took {minutes:.1f} min.", flush=True)
    if skipped:
        print(f"Skipped {skipped} crops with no valid ground truth.")
    return {"best_step": best["step"], "best_val_d1_noc": best["d1"], "stopped_at_step": stopped_at,
            "validation_curve_d1_noc": curve, "train_loss_curve": loss_curve, "lr_changes": lr_changes,
            "training_minutes": round(minutes, 1)}

print("Model, training and evaluation functions ready.")

In [ ]:
# ===== 9. Experiment definitions, reference numbers, results file =====
EXPERIMENTS = {
    "E1_bn_locked": {"kind": "full", "trainable": None, "lr": FULL_FT_LR,
                     "description": "Full fine-tuning (DepthAnything frozen), BatchNorm statistics locked"},
    "E2_gru_only": {"kind": "full", "trainable": GRU_MODULES, "lr": FULL_FT_LR,
                    "description": "Only GRU update block + upsampler trained, BatchNorm locked"},
    "E3a_lora_all": {"kind": "lora", "scope": None, "lr": LORA_LR,
                     "description": f"LoRA r={LORA_RANK} on all conv/linear layers outside DepthAnything, BatchNorm locked"},
    "E3b_lora_gru": {"kind": "lora", "scope": GRU_MODULES, "lr": LORA_LR,
                     "description": f"LoRA r={LORA_RANK} on GRU update block + upsampler, BatchNorm locked"},
}
unknown = [name for name in EXPERIMENTS_TO_RUN if name not in EXPERIMENTS]
if unknown:
    raise ValueError(f"Unknown experiments in EXPERIMENTS_TO_RUN: {unknown}")

# Earlier results with the same split and evaluation code (final_results.json)
REFERENCE = {
    "zero_shot": {
        "kitti_val":  {"all": {"epe": 0.8745, "d1": 2.7182, "mean_image_d1": 2.7636}, "noc": {"epe": 0.8661, "d1": 2.6866, "mean_image_d1": 2.7294}},
        "kitti_test": {"all": {"epe": 0.8545, "d1": 2.8561, "mean_image_d1": 2.9054}, "noc": {"epe": 0.8351, "d1": 2.7549, "mean_image_d1": 2.8204}},
        "middlebury": {"epe": 0.3634, "bp2": 1.0987},
    },
    "E0_full_ft": {
        "description": "Full fine-tuning (DepthAnything frozen), BatchNorm statistics drifting",
        "best_step": 350,
        "kitti_val":  {"all": {"epe": 0.7369, "d1": 1.4720, "mean_image_d1": 1.4598}, "noc": {"epe": 0.7292, "d1": 1.4421, "mean_image_d1": 1.4327}},
        "kitti_test": {"all": {"epe": 0.7343, "d1": 1.8623, "mean_image_d1": 1.7978}, "noc": {"epe": 0.7152, "d1": 1.7146, "mean_image_d1": 1.6744}},
        "middlebury": {"epe": 0.6345, "bp2": 2.4225},
    },
}

RESULTS_FILE = RESULTS_DIR / "e1_e4_lora_results.json"

def load_results():
    if RESULTS_FILE.exists():
        return json.loads(RESULTS_FILE.read_text())
    return {"protocol": "A", "seed": SEED, "reference": REFERENCE, "experiments": {}}

def save_results(results):
    tmp = RESULTS_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(results, indent=2))
    tmp.replace(RESULTS_FILE)

def pct_change(new, old):
    return 100.0 * (new / old - 1.0)

def print_experiment_report(name, entry):
    zs, e0 = REFERENCE["zero_shot"], REFERENCE["E0_full_ft"]
    print("\n" + "=" * 92)
    print(f"RESULTS {name}: {entry['description']}")
    print(f"trainable params {entry['trainable_parameters']:,} | best step {entry['best_step']} | "
          f"stopped at {entry['stopped_at_step']} | training {entry['training_minutes']} min")
    print("=" * 92)
    print(f"{'metric':<34}{'zero-shot':>12}{'E0 full FT':>12}{name:>16}{'vs zero-shot':>14}")
    lines = [
        ("KITTI15 val   D1-noc %", zs["kitti_val"]["noc"]["d1"], e0["kitti_val"]["noc"]["d1"], entry["kitti_val"]["noc"]["d1"]),
        ("KITTI15 test  D1-all %", zs["kitti_test"]["all"]["d1"], e0["kitti_test"]["all"]["d1"], entry["kitti_test"]["all"]["d1"]),
        ("KITTI15 test  D1-noc %", zs["kitti_test"]["noc"]["d1"], e0["kitti_test"]["noc"]["d1"], entry["kitti_test"]["noc"]["d1"]),
        ("KITTI15 test  EPE-all px", zs["kitti_test"]["all"]["epe"], e0["kitti_test"]["all"]["epe"], entry["kitti_test"]["all"]["epe"]),
        ("Middlebury    BP-2 %", zs["middlebury"]["bp2"], e0["middlebury"]["bp2"], entry["middlebury"]["bp2"]),
        ("Middlebury    EPE px", zs["middlebury"]["epe"], e0["middlebury"]["epe"], entry["middlebury"]["epe"]),
    ]
    for label, z, e, x in lines:
        print(f"{label:<34}{z:>12.4f}{e:>12.4f}{x:>16.4f}{pct_change(x, z):>+13.1f}%")
    print("(lower is better for every metric)")

results = load_results()
done = list(results["experiments"])
print("Results file:", RESULTS_FILE)
print("Already finished (will be skipped):", done if done else "none")
print("To run now:", [n for n in EXPERIMENTS_TO_RUN if n not in done] or "nothing")

In [ ]:
# ===== 10. Run E1, E2, E3a, E3b =====
for run_number, name in enumerate(EXPERIMENTS_TO_RUN, start=1):
    if name in results["experiments"]:
        print(f"\n### [{run_number}/{len(EXPERIMENTS_TO_RUN)}] {name}: already done, skipping")
        print_experiment_report(name, results["experiments"][name])
        continue
    spec = EXPERIMENTS[name]
    print("\n" + "#" * 92)
    print(f"### [{run_number}/{len(EXPERIMENTS_TO_RUN)}] {name}: {spec['description']}")
    print("#" * 92, flush=True)
    run_started = time.time()
    torch.cuda.empty_cache()
    torch.manual_seed(SEED)
    model, lora_layers = build_experiment_model(spec)
    trainable, total = count_parameters(model)
    print(f"Trainable parameters: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")
    if lora_layers:
        print(f"LoRA layers: {len(lora_layers)} (rank {LORA_RANK}, alpha {LORA_ALPHA}) in modules "
              f"{sorted({layer.split('.')[0] for layer in lora_layers})}")
    print(f"Learning rate {spec['lr']:g}", flush=True)

    # --- train (early stopping + LR scheduler inside) ---
    info = finetune(model, protocol_a["train"], protocol_a["val"], spec["lr"], SEED)

    # --- checkpoints ---
    if lora_layers:
        adapter = {k: v.detach().cpu() for k, v in model.state_dict().items() if ".lora_A." in k or ".lora_B." in k}
        adapter_path = CKPT_DIR / f"{name}_lora_adapter.pth"
        torch.save({"lora": adapter, "rank": LORA_RANK, "alpha": LORA_ALPHA, "scope": spec["scope"], **info}, adapter_path)
        print(f"Saved LoRA adapter ({adapter_path.stat().st_size / 1024**2:.1f} MiB): {adapter_path}")
        merged = merge_lora(model)
        model.eval()
        print(f"Merged {merged} LoRA layers into the base weights")
    if SAVE_CHECKPOINTS:
        ckpt_path = CKPT_DIR / f"{name}.pth"
        torch.save({"model": model.state_dict(), "experiment": name, "description": spec["description"], **info}, ckpt_path)
        print(f"Saved full checkpoint ({ckpt_path.stat().st_size / 1024**3:.2f} GiB): {ckpt_path}")
        print("  load with: model.load_state_dict(torch.load(path, weights_only=False)['model'])")

    # --- evaluation ---
    val_rows = kitti_metrics(model, protocol_a["val"], "KITTI 2015 validation")
    test_rows = kitti_metrics(model, protocol_a["test"], "KITTI 2015 held-out test")
    middlebury_rows = middlebury_metrics(model, name)
    entry = {
        "description": spec["description"],
        "kind": spec["kind"],
        "lr": spec["lr"],
        "trainable_modules": spec.get("trainable", spec.get("scope")) or "all except feature.dino",
        "lora": {"rank": LORA_RANK, "alpha": LORA_ALPHA, "layers": len(lora_layers)} if lora_layers else None,
        "batchnorm_locked": True,
        "trainable_parameters": trainable,
        "total_parameters": total,
        **info,
        "kitti_val": aggregate_kitti(val_rows),
        "kitti_test": aggregate_kitti(test_rows),
        "middlebury": aggregate_middlebury(middlebury_rows),
        "total_minutes": round((time.time() - run_started) / 60, 1),
    }
    summarize(entry["kitti_val"], f"{name} fine-tuned validation")
    summarize(entry["kitti_test"], f"{name} fine-tuned held-out test")
    print(f"{name} Middlebury weighted EPE: {entry['middlebury']['epe']:.4f}")
    print(f"{name} Middlebury weighted BP-2: {entry['middlebury']['bp2']:.4f}%")

    (RESULTS_DIR / f"{name}_rows.json").write_text(json.dumps(
        {"kitti_val": val_rows, "kitti_test": test_rows, "middlebury": middlebury_rows}, indent=1))
    results["experiments"][name] = entry
    save_results(results)
    print_experiment_report(name, entry)
    print(f"Saved results to {RESULTS_FILE} (run took {entry['total_minutes']} min)", flush=True)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll requested experiments finished.")

In [ ]:
# ===== 11. Final comparison table =====
import pandas as pd

def table_row(label, entry, trainable=None, best_step=None):
    zs = REFERENCE["zero_shot"]
    test, middlebury = entry["kitti_test"], entry["middlebury"]
    return {
        "model": label,
        "trainable params": trainable,
        "best step": best_step,
        "KITTI15 test D1-all %": test["all"]["d1"],
        "KITTI15 test D1-noc %": test["noc"]["d1"],
        "KITTI15 test EPE": test["all"]["epe"],
        "Middlebury BP-2 %": middlebury["bp2"],
        "Middlebury EPE": middlebury["epe"],
        "KITTI D1 vs zero-shot %": round(pct_change(test["all"]["d1"], zs["kitti_test"]["all"]["d1"]), 1),
        "Middlebury BP-2 vs zero-shot %": round(pct_change(middlebury["bp2"], zs["middlebury"]["bp2"]), 1),
    }

e1 = results["experiments"].get("E1_bn_locked", {})
rows = [table_row("zero-shot", REFERENCE["zero_shot"], 0, None),
        table_row("E0 full FT (BN drifting)", REFERENCE["E0_full_ft"], e1.get("trainable_parameters"), 350)]
rows += [table_row(name, entry, entry["trainable_parameters"], entry["best_step"])
         for name, entry in results["experiments"].items()]
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
print("Protocol A: KITTI 2015 held-out test (160 pairs) and Middlebury trainingH (15 scenes, non-occluded)\n")
print(comparison.to_string(index=False))
print("\nLower is better. 'vs zero-shot' columns: negative = better than the pretrained model.")
print("Saved:", RESULTS_DIR / "comparison_table.csv")

In [ ]:
# ===== 12. Download results and checkpoints =====
staging = Path("/content/e1_e4_lora_results")
shutil.rmtree(staging, ignore_errors=True)
staging.mkdir()
for file in RESULTS_DIR.glob("*"):
    if file.suffix in {".json", ".csv"}:
        shutil.copy2(file, staging / file.name)
for adapter in CKPT_DIR.glob("*_lora_adapter.pth"):
    shutil.copy2(adapter, staging / adapter.name)   # small (a few MiB)
results_zip = shutil.make_archive("/content/foundationstereo_e1_e4_lora_results", "zip", staging)
print("Results zip (JSON + CSV + LoRA adapters):", results_zip)

print("\nCheckpoints:")
for ckpt in sorted(CKPT_DIR.glob("*.pth")):
    print(f"  {ckpt}  ({ckpt.stat().st_size / 1024**2:,.0f} MiB)")
if DRIVE_OK:
    print("Checkpoints and results are in Google Drive:", RESULTS_DIR)

try:
    from google.colab import files
    files.download(results_zip)
    if DOWNLOAD_FULL_CHECKPOINTS:
        for ckpt in sorted(CKPT_DIR.glob("*.pth")):
            if not ckpt.name.endswith("_lora_adapter.pth"):
                print("Downloading", ckpt.name, flush=True)
                files.download(str(ckpt))
    else:
        print("Full checkpoints not pushed through the browser (DOWNLOAD_FULL_CHECKPOINTS = False);",
              "get them from Google Drive or the Colab file panel.")
except Exception as error:
    print("Browser download not available:", error)